# Multi-Layer Perceptron (MLP) from Scratch

In this notebook we build a small neural network **by hand**, one neuron at a time, using only plain Python (and `numpy` for `exp`).
No classes, no frameworks. The goal is that you can *see* what goes into every neuron and what comes out.


**Our network**

![3 layer MLP](images/ann_2-3-2-1.png)

Every neuron does the same two things:

1. **Weighted sum:** `z = (w1 * input1) + (w2 * input2) + ... + b`
2. **Activation:** `a = sigmoid(z)`  (squashes `z` into a number between 0 and 1)

**Naming convention used everywhere**

| Name | Meaning |
|---|---|
| `w1_21` | weight in **layer 1**, going into **node 2**, coming from **input 1** |
| `b2_1` | bias of **layer 2**, **node 1** |
| `z1_3` | weighted sum (before activation) of layer 1, node 3 |
| `a1_3` | output (after activation) of layer 1, node 3 |

**Total trainable parameters:** layer 1 → 3×2 weights + 3 biases = 9, layer 2 → 2×3 weights + 2 biases = 8, output → 1×2 weights + 1 bias = 3. That is **20 parameters**.

## Step 0: Imports and the activation function

In [1]:
import numpy as np

def sigmoid(z):
    # squashes any number into the range (0, 1)
    return 1 / (1 + np.exp(-z))

**Sigmoid** is `σ(z) = 1 / (1 + e^(-z))`.

Its derivative has a very handy form that we will need in backprop:

`σ'(z) = σ(z) · (1 − σ(z)) = a · (1 − a)`  (where `a = σ(z)`)

## Step 1: Our data and starting parameters

To keep the numbers easy to follow we use **one training example**:

- inputs: `x1 = 0.5`, `x2 = 0.8`
- true answer (target): `y = 1`

The weights and biases below are small, arbitrary starting values. (Normally they are random.)

In [2]:
# ---------- one training example ----------
x1 = 0.5
x2 = 0.8
y  = 1          # the true answer we want the network to predict

# ---------- Layer 1 parameters (3 neurons, each receives x1 and x2) ----------
w1_11, w1_12, b1_1 =  0.2, -0.3,  0.10    # neuron 1
w1_21, w1_22, b1_2 =  0.4,  0.1, -0.10    # neuron 2
w1_31, w1_32, b1_3 = -0.5,  0.6,  0.05    # neuron 3

# ---------- Layer 2 parameters (2 neurons, each receives a1_1, a1_2, a1_3) ----------
w2_11, w2_12, w2_13, b2_1 =  0.3, -0.2, 0.5, 0.00   # neuron 1
w2_21, w2_22, w2_23, b2_2 = -0.4,  0.6, 0.1, 0.10   # neuron 2

# ---------- Output layer parameters (1 neuron, receives a2_1, a2_2) ----------
w3_1, w3_2, b3 = 0.7, -0.5, 0.20

# ---------- learning rate ----------
lr = 0.5

# PART A: Forward Propagation

Data flows left to right: **inputs → layer 1 → layer 2 → output**.
For each neuron we print what comes **in** (inputs, weights, bias) and what goes **out**.

## Step 2: Hidden Layer 1 (3 neurons)

Each neuron receives **both inputs** `x1, x2`.

```
z1_1 = w1_11·x1 + w1_12·x2 + b1_1        a1_1 = sigmoid(z1_1)
z1_2 = w1_21·x1 + w1_22·x2 + b1_2        a1_2 = sigmoid(z1_2)
z1_3 = w1_31·x1 + w1_32·x2 + b1_3        a1_3 = sigmoid(z1_3)
```

In [3]:
# ----- Layer 1, neuron 1 -----
z1_1 = w1_11 * x1 + w1_12 * x2 + b1_1
a1_1 = sigmoid(z1_1)
print(f"L1 neuron 1 | IN : x1={x1}, x2={x2} | weights: w1_11={w1_11:.4f}, w1_12={w1_12:.4f} | bias: b1_1={b1_1:.4f}")
print(f"            | OUT: z1_1={z1_1:.4f}  ->  a1_1={a1_1:.4f}")

# ----- Layer 1, neuron 2 -----
z1_2 = w1_21 * x1 + w1_22 * x2 + b1_2
a1_2 = sigmoid(z1_2)
print(f"L1 neuron 2 | IN : x1={x1}, x2={x2} | weights: w1_21={w1_21:.4f}, w1_22={w1_22:.4f} | bias: b1_2={b1_2:.4f}")
print(f"            | OUT: z1_2={z1_2:.4f}  ->  a1_2={a1_2:.4f}")

# ----- Layer 1, neuron 3 -----
z1_3 = w1_31 * x1 + w1_32 * x2 + b1_3
a1_3 = sigmoid(z1_3)
print(f"L1 neuron 3 | IN : x1={x1}, x2={x2} | weights: w1_31={w1_31:.4f}, w1_32={w1_32:.4f} | bias: b1_3={b1_3:.4f}")
print(f"            | OUT: z1_3={z1_3:.4f}  ->  a1_3={a1_3:.4f}")

L1 neuron 1 | IN : x1=0.5, x2=0.8 | weights: w1_11=0.2000, w1_12=-0.3000 | bias: b1_1=0.1000
            | OUT: z1_1=-0.0400  ->  a1_1=0.4900
L1 neuron 2 | IN : x1=0.5, x2=0.8 | weights: w1_21=0.4000, w1_22=0.1000 | bias: b1_2=-0.1000
            | OUT: z1_2=0.1800  ->  a1_2=0.5449
L1 neuron 3 | IN : x1=0.5, x2=0.8 | weights: w1_31=-0.5000, w1_32=0.6000 | bias: b1_3=0.0500
            | OUT: z1_3=0.2800  ->  a1_3=0.5695


## Step 3: Hidden Layer 2 (2 neurons)

Now the **inputs are the outputs of layer 1**: `a1_1, a1_2, a1_3`. Each neuron receives all three.

```
z2_1 = w2_11·a1_1 + w2_12·a1_2 + w2_13·a1_3 + b2_1        a2_1 = sigmoid(z2_1)
z2_2 = w2_21·a1_1 + w2_22·a1_2 + w2_23·a1_3 + b2_2        a2_2 = sigmoid(z2_2)
```

In [4]:
# ----- Layer 2, neuron 1 -----
z2_1 = w2_11 * a1_1 + w2_12 * a1_2 + w2_13 * a1_3 + b2_1
a2_1 = sigmoid(z2_1)
print(f"L2 neuron 1 | IN : a1_1={a1_1:.4f}, a1_2={a1_2:.4f}, a1_3={a1_3:.4f}")
print(f"            | weights: w2_11={w2_11:.4f}, w2_12={w2_12:.4f}, w2_13={w2_13:.4f} | bias: b2_1={b2_1:.4f}")
print(f"            | OUT: z2_1={z2_1:.4f}  ->  a2_1={a2_1:.4f}")

# ----- Layer 2, neuron 2 -----
z2_2 = w2_21 * a1_1 + w2_22 * a1_2 + w2_23 * a1_3 + b2_2
a2_2 = sigmoid(z2_2)
print(f"L2 neuron 2 | IN : a1_1={a1_1:.4f}, a1_2={a1_2:.4f}, a1_3={a1_3:.4f}")
print(f"            | weights: w2_21={w2_21:.4f}, w2_22={w2_22:.4f}, w2_23={w2_23:.4f} | bias: b2_2={b2_2:.4f}")
print(f"            | OUT: z2_2={z2_2:.4f}  ->  a2_2={a2_2:.4f}")

L2 neuron 1 | IN : a1_1=0.4900, a1_2=0.5449, a1_3=0.5695
            | weights: w2_11=0.3000, w2_12=-0.2000, w2_13=0.5000 | bias: b2_1=0.0000
            | OUT: z2_1=0.3228  ->  a2_1=0.5800
L2 neuron 2 | IN : a1_1=0.4900, a1_2=0.5449, a1_3=0.5695
            | weights: w2_21=-0.4000, w2_22=0.6000, w2_23=0.1000 | bias: b2_2=0.1000
            | OUT: z2_2=0.2879  ->  a2_2=0.5715


## Step 4: Output Layer (1 neuron)

Inputs are the outputs of layer 2: `a2_1, a2_2`. The result is the network's **prediction** `y_hat`.

```
z3    = w3_1·a2_1 + w3_2·a2_2 + b3
y_hat = sigmoid(z3)
```

In [5]:
# ----- Output neuron -----
z3 = w3_1 * a2_1 + w3_2 * a2_2 + b3
y_hat = sigmoid(z3)
print(f"Output neuron | IN : a2_1={a2_1:.4f}, a2_2={a2_2:.4f} | weights: w3_1={w3_1}, w3_2={w3_2} | bias: b3={b3}")
print(f"              | OUT: z3={z3:.4f}  ->  y_hat={y_hat:.4f}")
print()
print(f"Prediction y_hat = {y_hat:.4f},  true answer y = {y}")

Output neuron | IN : a2_1=0.5800, a2_2=0.5715 | weights: w3_1=0.7, w3_2=-0.5 | bias: b3=0.2
              | OUT: z3=0.3203  ->  y_hat=0.5794

Prediction y_hat = 0.5794,  true answer y = 1


# PART B: The Loss

The prediction is not equal to the target, so we measure *how wrong* we are with the **squared error loss**:

`L = ½ · (y − y_hat)²`

(The ½ is just there so the derivative comes out clean: `dL/dy_hat = −(y − y_hat) = y_hat − y`.)

In [6]:
loss = 0.5 * (y - y_hat) ** 2
print(f"Loss before training = {loss:.6f}")

Loss before training = 0.088457


# PART C: Backpropagation

**Goal:** find how much each of the 20 parameters contributes to the loss, i.e. `dL/dw` and `dL/db` for each one.

**Idea:** go **backwards** (right to left) using the **chain rule**. Each layer reuses what the layer after it already computed.

For every neuron we compute a number called **delta**:

`delta = dL/dz`  (how much the loss changes if this neuron's `z` changes)

Once we know a neuron's delta, its parameter gradients are easy:

```
dL/dw = delta × (the input that this weight multiplied)
dL/db = delta
```

## Step 5: Output layer gradients

**Formulas**

```
dL/dy_hat = y_hat − y
delta3    = dL/dz3 = (y_hat − y) · y_hat · (1 − y_hat)         [chain rule: dL/dy_hat × dy_hat/dz3]

dL/dw3_1  = delta3 · a2_1
dL/dw3_2  = delta3 · a2_2
dL/db3    = delta3
```

In [7]:
dL_dyhat = y_hat - y
delta3 = dL_dyhat * y_hat * (1 - y_hat)

dL_dw3_1 = delta3 * a2_1
dL_dw3_2 = delta3 * a2_2
dL_db3   = delta3

print(f"delta3   = {delta3:.6f}")
print(f"dL/dw3_1 = {dL_dw3_1:.6f}")
print(f"dL/dw3_2 = {dL_dw3_2:.6f}")
print(f"dL/db3   = {dL_db3:.6f}")

delta3   = -0.102502
dL/dw3_1 = -0.059452
dL/dw3_2 = -0.058577
dL/db3   = -0.102502


## Step 6: Hidden layer 2 gradients

First we pass the error **back** through the output weights. Neuron `a2_k` affected the loss through `z3`, via weight `w3_k`:

```
dL/da2_1 = delta3 · w3_1                      [chain rule=> dL/da2_1 = dL/dy_hat × dy_hat/dz3 × dz3/da2_1]
dL/da2_2 = delta3 · w3_2
```

Then through the sigmoid of that neuron:

```
delta2_1 = dL/da2_1 · a2_1 · (1 − a2_1)    [chain rule=>delta2_1 = dL/dy_hat × dy_hat/dz3 × dz3/da2_1 × da2_1/dz2_1]
delta2_2 = dL/da2_2 · a2_2 · (1 − a2_2)
```

Then the parameter gradients (inputs of layer 2 are `a1_1, a1_2, a1_3`):

```
dL/dw2_11 = delta2_1 · a1_1     dL/dw2_12 = delta2_1 · a1_2     dL/dw2_13 = delta2_1 · a1_3     dL/db2_1 = delta2_1
dL/dw2_21 = delta2_2 · a1_1     dL/dw2_22 = delta2_2 · a1_2     dL/dw2_23 = delta2_2 · a1_3     dL/db2_2 = delta2_2
```

In [8]:
# error flowing back into layer 2 outputs
dL_da2_1 = delta3 * w3_1
dL_da2_2 = delta3 * w3_2

# through the sigmoid of each layer-2 neuron
delta2_1 = dL_da2_1 * a2_1 * (1 - a2_1)
delta2_2 = dL_da2_2 * a2_2 * (1 - a2_2)

# parameter gradients
dL_dw2_11 = delta2_1 * a1_1
dL_dw2_12 = delta2_1 * a1_2
dL_dw2_13 = delta2_1 * a1_3
dL_db2_1   = delta2_1

dL_dw2_21 = delta2_2 * a1_1
dL_dw2_22 = delta2_2 * a1_2
dL_dw2_23 = delta2_2 * a1_3
dL_db2_2   = delta2_2

print(f"delta2_1 = {delta2_1:.6f}, delta2_2 = {delta2_2:.6f}")
print(f"dL_dw2_11={dL_dw2_11:.6f}, dL_dw2_12={dL_dw2_12:.6f}, dL_dw2_13={dL_dw2_13:.6f}, dL_db2_1={dL_db2_1:.6f}")
print(f"dL_dw2_21={dL_dw2_21:.6f}, dL_dw2_22={dL_dw2_22:.6f}, dL_dw2_23={dL_dw2_23:.6f}, dL_db2_2={dL_db2_2:.6f}")

delta2_1 = -0.017479, delta2_2 = 0.012551
dL_dw2_11=-0.008565, dL_dw2_12=-0.009524, dL_dw2_13=-0.009955, dL_db2_1=-0.017479
dL_dw2_21=0.006150, dL_dw2_22=0.006839, dL_dw2_23=0.007148, dL_db2_2=0.012551


## Step 7: Hidden layer 1 gradients

Each layer-1 neuron `a1_j` feeds **both** layer-2 neurons, so the error comes back along **two paths** and we **add** them:

```
dL/da1_1 = delta2_1 · w2_11 + delta2_2 · w2_21
dL/da1_2 = delta2_1 · w2_12 + delta2_2 · w2_22
dL/da1_3 = delta2_1 · w2_13 + delta2_2 · w2_23
```

Through the sigmoid:

```
delta1_1 = dL/da1_1 · a1_1 · (1 − a1_1)
delta1_2 = dL/da1_2 · a1_2 · (1 − a1_2)
delta1_3 = dL/da1_3 · a1_3 · (1 − a1_3)
```

Parameter gradients (inputs of layer 1 are `x1, x2`):

```
dL/dw1_11 = delta1_1 · x1     dL/dw1_12 = delta1_1 · x2     dL/db1_1 = delta1_1
dL/dw1_21 = delta1_2 · x1     dL/dw1_22 = delta1_2 · x2     dL/db1_2 = delta1_2
dL/dw1_31 = delta1_3 · x1     dL/dw1_32 = delta1_3 · x2     dL/db1_3 = delta1_3
```

In [9]:
# error flowing back into layer 1 outputs (sum over the two paths)
dL_da1_1 = delta2_1 * w2_11 + delta2_2 * w2_21
dL_da1_2 = delta2_1 * w2_12 + delta2_2 * w2_22
dL_da1_3 = delta2_1 * w2_13 + delta2_2 * w2_23

# through the sigmoid of each layer-1 neuron
delta1_1 = dL_da1_1 * a1_1 * (1 - a1_1)
delta1_2 = dL_da1_2 * a1_2 * (1 - a1_2)
delta1_3 = dL_da1_3 * a1_3 * (1 - a1_3)

# parameter gradients
dL_dw1_11 = delta1_1 * x1
dL_dw1_12 = delta1_1 * x2
dL_db1_1   = delta1_1

dL_dw1_21 = delta1_2 * x1
dL_dw1_22 = delta1_2 * x2
dL_db1_2   = delta1_2

dL_dw1_31 = delta1_3 * x1
dL_dw1_32 = delta1_3 * x2
dL_db1_3   = delta1_3

print(f"delta1_1 = {delta1_1:.6f}, delta1_2 = {delta1_2:.6f}, delta1_3 = {delta1_3:.6f}")
print(f"dL_dw1_11={dL_dw1_11:.6f}, dL_dw1_12={dL_dw1_12:.6f}, dL_db1_1={dL_db1_1:.6f}")
print(f"dL_dw1_21={dL_dw1_21:.6f}, dL_dw1_22={dL_dw1_22:.6f}, dL_db1_2={dL_db1_2:.6f}")
print(f"dL_dw1_31={dL_dw1_31:.6f}, dL_dw1_32={dL_dw1_32:.6f}, dL_db1_3={dL_db1_3:.6f}")

delta1_1 = -0.002565, delta1_2 = 0.002734, delta1_3 = -0.001835
dL_dw1_11=-0.001282, dL_dw1_12=-0.002052, dL_db1_1=-0.002565
dL_dw1_21=0.001367, dL_dw1_22=0.002187, dL_db1_2=0.002734
dL_dw1_31=-0.000917, dL_dw1_32=-0.001468, dL_db1_3=-0.001835


# PART D: Gradient Descent: Updating Every Parameter

The gradient tells us which direction **increases** the loss, so we step in the **opposite** direction:

`new_param = old_param − learning_rate × dL/dparam`

We use `lr = 0.5`. Important: we computed **all** gradients first (using the old weights), and only now do we update.

In [10]:
# ---------- Output layer ----------
w3_1 = w3_1 - lr * dL_dw3_1
w3_2 = w3_2 - lr * dL_dw3_2
b3   = b3   - lr * dL_db3

# ---------- Hidden layer 2 ----------
w2_11 = w2_11 - lr * dL_dw2_11
w2_12 = w2_12 - lr * dL_dw2_12
w2_13 = w2_13 - lr * dL_dw2_13
b2_1   = b2_1   - lr * dL_db2_1
w2_21 = w2_21 - lr * dL_dw2_21
w2_22 = w2_22 - lr * dL_dw2_22
w2_23 = w2_23 - lr * dL_dw2_23
b2_2   = b2_2   - lr * dL_db2_2

# ---------- Hidden layer 1 ----------
w1_11 = w1_11 - lr * dL_dw1_11
w1_12 = w1_12 - lr * dL_dw1_12
b1_1   = b1_1   - lr * dL_db1_1
w1_21 = w1_21 - lr * dL_dw1_21
w1_22 = w1_22 - lr * dL_dw1_22
b1_2   = b1_2   - lr * dL_db1_2
w1_31 = w1_31 - lr * dL_dw1_31
w1_32 = w1_32 - lr * dL_dw1_32
b1_3   = b1_3   - lr * dL_db1_3

print("All 20 parameters updated!")

All 20 parameters updated!


## Step 8: Did it work? Run forward propagation again

With the updated parameters the prediction should be **closer to 1** and the loss **smaller**.

In [11]:
# forward pass again with the NEW parameters (same formulas as Part A, shortened)
a1_1 = sigmoid(w1_11 * x1 + w1_12 * x2 + b1_1)
a1_2 = sigmoid(w1_21 * x1 + w1_22 * x2 + b1_2)
a1_3 = sigmoid(w1_31 * x1 + w1_32 * x2 + b1_3)

a2_1 = sigmoid(w2_11 * a1_1 + w2_12 * a1_2 + w2_13 * a1_3 + b2_1)
a2_2 = sigmoid(w2_21 * a1_1 + w2_22 * a1_2 + w2_23 * a1_3 + b2_2)

y_hat_new = sigmoid(w3_1 * a2_1 + w3_2 * a2_2 + b3)
loss_new = 0.5 * (y - y_hat_new) ** 2

print(f"Before: y_hat = {y_hat:.4f}, loss = {loss:.6f}")
print(f"After : y_hat = {y_hat_new:.4f}, loss = {loss_new:.6f}")

Before: y_hat = 0.5794, loss = 0.088457
After : y_hat = 0.6011, loss = 0.079576


# PART E: Training = Repeating Parts A to D

One forward pass + one backward pass + one update is **one training step**. Training simply repeats this many times.

Below is the **same math** as above, written compactly with `numpy` arrays so we can loop. A row of a weight matrix holds the weights of one neuron, so `W1[0]` is `[w1_11, w1_12]`.
We restart from the same initial values.

In [12]:
x = np.array([0.5, 0.8])
y = 1

W1 = np.array([[ 0.2, -0.3], [0.4, 0.1], [-0.5, 0.6]]);  b1 = np.array([0.10, -0.10, 0.05])
W2 = np.array([[ 0.3, -0.2, 0.5], [-0.4, 0.6, 0.1]]);    b2 = np.array([0.00, 0.10])
W3 = np.array([0.7, -0.5]);                              b3 = 0.20
lr = 0.5

for epoch in range(1, 501):
    # ---- forward ----
    a1 = sigmoid(W1 @ x + b1)
    a2 = sigmoid(W2 @ a1 + b2)
    y_hat = sigmoid(W3 @ a2 + b3)
    loss = 0.5 * (y - y_hat) ** 2

    # ---- backward (same formulas as Part C) ----
    delta3 = (y_hat - y) * y_hat * (1 - y_hat)
    delta2 = (delta3 * W3) * a2 * (1 - a2)
    delta1 = (W2.T @ delta2) * a1 * (1 - a1)

    # ---- update (same formulas as Part D) ----
    W3 = W3 - lr * delta3 * a2;            b3 = b3 - lr * delta3
    W2 = W2 - lr * np.outer(delta2, a1);   b2 = b2 - lr * delta2
    W1 = W1 - lr * np.outer(delta1, x);    b1 = b1 - lr * delta1

    if epoch in (1, 2, 10, 50, 100, 200, 500):
        print(f"epoch {epoch:3d} | y_hat = {y_hat:.4f} | loss = {loss:.6f}")

epoch   1 | y_hat = 0.5794 | loss = 0.088457
epoch   2 | y_hat = 0.6011 | loss = 0.079576
epoch  10 | y_hat = 0.7206 | loss = 0.039035
epoch  50 | y_hat = 0.8749 | loss = 0.007831
epoch 100 | y_hat = 0.9163 | loss = 0.003507
epoch 200 | y_hat = 0.9438 | loss = 0.001577
epoch 500 | y_hat = 0.9665 | loss = 0.000562


Notice that after epoch 1 the loss matches the "Before" value from Step 8, and the prediction keeps creeping toward the target `1`. That is learning!

**Recap**

1. **Forward:** `z = w·input + b`, then `a = sigmoid(z)`, layer by layer.
2. **Loss:** `L = ½(y − y_hat)²`.
3. **Backward:** compute `delta` for each neuron from right to left, then `dL/dw = delta × input`, `dL/db = delta`.
4. **Update:** `param = param − lr × gradient`.
5. Repeat.